# Model Training for Predictive Analytics
**Development of a predictive analytics model for predicting sought-after professions**

This notebook covers:
- Task definition and target variables
- Model comparison and selection
- Hyperparameter tuning
- Model evaluation
- Feature importance analysis

## 1. Setup and Imports

In [ ]:
import sys
sys.path.append('..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Custom modules
from src.preprocessing import VacancyPreprocessor
from src.features import FeatureEngineer
from src.models import ProfessionClassifier, ModelComparator, SalaryPredictor

# Sklearn
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix

# Settings
pd.set_option('display.max_columns', None)
sns.set_style('whitegrid')

print("✅ Libraries imported!")

## 2. Define ML Tasks

### Task Definition

**Primary Task: Multi-class Classification**
- **Target**: Job (profession category)
- **Goal**: Predict profession based on skills, experience, location
- **Classes**: 14 profession categories

**Secondary Task: Regression** (optional)
- **Target**: salary_numeric
- **Goal**: Predict salary based on features
- **Challenge**: Many missing values (~70%)

## 3. Data Preparation

In [ ]:
# Load and preprocess data
preprocessor = VacancyPreprocessor()
df = preprocessor.load_data('../results.csv')
df_processed = preprocessor.preprocess()

# Feature engineering
fe = FeatureEngineer()
X, y, feature_columns = fe.prepare_for_modeling(
    df_processed,
    target_column='Job',
    use_tfidf=True,
    use_skills=True,
    tfidf_max_features=50
)

print(f"\nTarget classes: {len(np.unique(y))}")
print(f"Class distribution:")
print(pd.Series(y).value_counts())

In [ ]:
# Train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, 
    test_size=0.2, 
    random_state=42,
    stratify=y
)

print(f"Training: {X_train.shape[0]} samples")
print(f"Test: {X_test.shape[0]} samples")

## 4. Model Comparison

In [ ]:
# Initialize comparator
comparator = ModelComparator()

# Add models
comparator.add_model('Random Forest', ProfessionClassifier('random_forest'))
comparator.add_model('Logistic Regression', ProfessionClassifier('logistic'))
comparator.add_model('Gradient Boosting', ProfessionClassifier('gradient_boosting'))
comparator.add_model('KNN', ProfessionClassifier('knn'))

# Compare
results = comparator.compare(X_train, y_train, X_test, y_test, cv=5)

In [ ]:
# Visualize results
fig, ax = plt.subplots(figsize=(12, 6))

results_sorted = results.sort_values('Test_F1', ascending=True)

colors = ['lightcoral' if 'Train' in col else 'steelblue' for col in ['Train_F1', 'Test_F1']]
x = np.arange(len(results_sorted))
width = 0.35

bars1 = ax.barh(x - width/2, results_sorted['Train_F1'], width, label='Train F1', color='lightcoral')
bars2 = ax.barh(x + width/2, results_sorted['Test_F1'], width, label='Test F1', color='steelblue')

ax.set_yticks(x)
ax.set_yticklabels(results_sorted['Model'])
ax.set_xlabel('F1 Score')
ax.set_title('Model Comparison: Train vs Test F1 Score', fontsize=14)
ax.legend()
ax.set_xlim(0, 1)

plt.tight_layout()
plt.show()

## 5. Best Model Selection and Tuning

In [ ]:
# Get best model
best_name, best_model = comparator.get_best_model()
print(f"Best model: {best_name}")

# Classification report
target_names = [fe.label_encoders['target'].classes_[i] for i in range(len(fe.label_encoders['target'].classes_))]
print("\n=== Classification Report ===")
print(best_model.get_classification_report(X_test, y_test, target_names=target_names))

In [ ]:
# Confusion matrix
cm = best_model.get_confusion_matrix(X_test, y_test)

plt.figure(figsize=(14, 10))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=target_names, yticklabels=target_names)
plt.title('Confusion Matrix', fontsize=14)
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

## 6. Feature Importance

In [ ]:
# Get feature importance (for tree-based models)
if hasattr(best_model.model, 'feature_importances_'):
    importance = best_model.model.feature_importances_
    
    feat_importance = pd.DataFrame({
        'Feature': feature_columns,
        'Importance': importance
    }).sort_values('Importance', ascending=False).head(20)
    
    plt.figure(figsize=(12, 8))
    sns.barplot(data=feat_importance, x='Importance', y='Feature', palette='viridis')
    plt.title('Top-20 Feature Importances', fontsize=14)
    plt.tight_layout()
    plt.show()
    
    print("\nTop 10 Most Important Features:")
    print(feat_importance.head(10).to_string(index=False))
else:
    print("Feature importance not available for this model type")

## 7. Hyperparameter Tuning

In [ ]:
# Tune best model
print(f"Tuning {best_name}...")

tuned_model = ProfessionClassifier(best_name)
tuned_model.hyperparameter_search(
    X_train, y_train,
    search_type='random',
    cv=3,
    n_iter=10
)

# Evaluate tuned model
tuned_metrics = tuned_model.evaluate(X_test, y_test)
print(f"\nTuned model metrics:")
for metric, value in tuned_metrics.items():
    print(f"   {metric}: {value:.4f}")

## 8. Save Final Model

In [ ]:
import joblib
import os

os.makedirs('../models', exist_ok=True)

# Save model
joblib.dump(tuned_model.model, '../models/best_classifier.joblib')
joblib.dump(fe.label_encoders, '../models/label_encoders.joblib')

print("✅ Model saved to models/best_classifier.joblib")
print("✅ Encoders saved to models/label_encoders.joblib")

## 9. Summary

In [ ]:
print("=" * 60)
print("MODEL TRAINING SUMMARY")
print("=" * 60)

print(f"\n📊 Dataset:")
print(f"   • Total samples: {len(X)}")
print(f"   • Features: {len(feature_columns)}")
print(f"   • Classes: {len(np.unique(y))}")

print(f"\n🏆 Best Model: {best_name}")
print(f"   • Test Accuracy: {tuned_metrics['accuracy']:.4f}")
print(f"   • Test F1: {tuned_metrics['f1']:.4f}")
print(f"   • Test Precision: {tuned_metrics['precision']:.4f}")
print(f"   • Test Recall: {tuned_metrics['recall']:.4f}")

print(f"\n📁 Saved Files:")
print(f"   • models/best_classifier.joblib")
print(f"   • models/label_encoders.joblib")

print("\n" + "=" * 60)
print("✅ MODEL TRAINING COMPLETE!")
print("=" * 60)